# ☁️ Understanding Cloud Computing — Showcase

> **Platform:** DataCamp · **Track:** Data Engineer in Python · **Course 1**

This is a conceptual course (no coding exercises). This notebook is my own synthesis of the theory, with short Python snippets that turn each concept into something concrete: a responsibility matrix, a scaling cost model, a PII check and a provider lookup.

**Objectives**
1. Explain what cloud computing is and why it creates value.
2. Compare service models (IaaS, PaaS, SaaS, FaaS) and deployment models (public, private, hybrid, multicloud, community).
3. Understand how regulations (GDPR, personal data) affect where data lives.
4. Map the main cloud roles and the Big Three providers' core services.

In [1]:
import pandas as pd

## 1. What is cloud computing?

Cloud computing is the delivery of compute, storage and databases over the internet instead of running them on servers we own (**on-premise**). Three ideas make it valuable:

- **Virtualization** — one physical server is split into many independent virtual servers, so hardware is used more efficiently.
- **Scalability** — *vertical* (add CPU/RAM to one machine) or *horizontal* (add more machines and spread the traffic). **Elasticity** means doing this automatically with demand.
- **Pay-as-you-go** — we pay for what we consume instead of a large upfront capital expense (CapEx).

Other benefits: speed to market, up-to-date hardware, global reach, data replication across data centers and support for compliance.

### Scaling in practice: fixed capacity vs. pay-as-you-go

A simple model of an e-commerce site with a Black Friday spike. On-premise must be sized for the peak all year; the cloud scales horizontally and bills per server-hour.

In [2]:
# Servers needed per month (traffic spike in November)
demand = pd.Series([4, 4, 5, 4, 5, 5, 6, 5, 5, 6, 20, 12],
                   index=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])

COST_PER_SERVER_MONTH = 100  # illustrative unit cost

on_prem = demand.max() * COST_PER_SERVER_MONTH * len(demand)   # sized for the peak, all year
cloud = (demand * COST_PER_SERVER_MONTH).sum()                  # pay only for what is used

print(f"On-premise (sized for peak): ${on_prem:,}")
print(f"Cloud (pay-as-you-go):       ${cloud:,}")
print(f"Idle capacity paid on-prem:  {1 - demand.sum() / (demand.max() * len(demand)):.0%}")

On-premise (sized for peak): $24,000
Cloud (pay-as-you-go):       $8,100
Idle capacity paid on-prem:  66%


**Interpretation:** with spiky demand, most on-premise capacity sits idle. This is why companies with strong seasonal peaks benefit most from a highly scalable cloud infrastructure.

## 2. Cloud service models

The course uses a transport analogy: the more the provider manages, the less we control.

| Model | Analogy | We manage | Examples |
|---|---|---|---|
| On-premise | Own a car | Everything | Own data center |
| IaaS | Rent a car | OS, runtime, apps, data | AWS EC2, Google Compute Engine |
| PaaS | Take an Uber | Apps and data | Google App Engine, AWS Elastic Beanstalk |
| SaaS | Take the bus | Nothing (just use it) | Google Workspace, Dropbox, Office 365 |

Two more models: **FaaS / serverless** (run single functions, pay per execution) and **XaaS** (anything delivered as a service).

In [3]:
layers = ["Networking", "Storage", "Servers", "Virtualization", "OS", "Runtime", "Application", "Data"]
provider_manages = {"On-premise": 0, "IaaS": 4, "PaaS": 6, "SaaS": 8}  # layers handled by the provider, bottom-up

matrix = pd.DataFrame(
    {model: ["Provider" if i < n else "You" for i in range(len(layers))]
     for model, n in provider_manages.items()},
    index=layers,
)
matrix

,On-premise,IaaS,PaaS,SaaS
Networking,You,Provider,Provider,Provider
Storage,You,Provider,Provider,Provider
Servers,You,Provider,Provider,Provider
Virtualization,You,Provider,Provider,Provider
OS,You,You,Provider,Provider
Runtime,You,You,Provider,Provider
Application,You,You,You,Provider
Data,You,You,You,Provider


**Interpretation:** a team that wants infrastructure on demand but full control over its applications fits **IaaS**; a business that only wants ready-to-use software (inventory, orders, payments) fits **SaaS**.

## 3. Deployment models

The choice depends on how much control the business needs, its security requirements and its budget.

- **Private cloud** — exclusive to one organization; highest control, but more time and CapEx. Unlike on-premise, it still uses cloud principles (virtualization, on-demand resources) and can be hosted by a third party.
- **Public cloud** — shared infrastructure owned by a provider (AWS, Azure…) and accessed over the internet; fast to start, low upfront cost, no physical access to data centers.
- **Hybrid cloud** — two or more deployment models connected, e.g. sensitive patient data in a private cloud and BI tools in the public cloud. Enables **cloud bursting**: overflow traffic goes to the public cloud when the private one is full.
- **Multicloud** — services from *several providers* (e.g. Azure for backups, AWS for hosting, Google Cloud for analytics). Not the same as hybrid, which combines *deployment models*.
- **Community cloud** — shared by organizations with a common mission or regulation (e.g. government agencies sharing tax data).

In [4]:
def suggest_deployment(exclusive_infra: bool, shared_with_peers: bool,
                       burst_to_public: bool, several_providers: bool) -> str:
    """Toy decision rule that encodes the course's definitions."""
    if shared_with_peers:
        return "Community cloud"
    if several_providers:
        return "Multicloud"
    if exclusive_infra and burst_to_public:
        return "Hybrid cloud"
    if exclusive_infra:
        return "Private cloud"
    return "Public cloud"

cases = {
    "Hospital: private patient data + public BI, bursting at peaks": (True, False, True, False),
    "Government agencies sharing tax data": (False, True, False, False),
    "Startup launching fast with low upfront cost": (False, False, False, False),
    "Backups on Azure, hosting on AWS, analytics on GCP": (False, False, False, True),
}
for case, args in cases.items():
    print(f"{suggest_deployment(*args):<16} <- {case}")

Hybrid cloud     <- Hospital: private patient data + public BI, bursting at peaks
Community cloud  <- Government agencies sharing tax data
Public cloud     <- Startup launching fast with low upfront cost
Multicloud       <- Backups on Azure, hosting on AWS, analytics on GCP


**Interpretation:** there is no universally best model. Multiple agencies with a shared mission point to a **community** model, not hybrid, which was a key distinction in the course.

## 4. Regulations and personal data

Global data centers reduce **latency** (the delay between request and response) by serving users from the nearest region. But moving data across borders means complying with the rules of the origin country, the data-center location and the company's home country.

**GDPR** (EU) regulates collecting, processing and storing personal data of people in the EU: explicit consent, timely breach notification, encryption/anonymization/pseudonymization, limits on transferring data outside the EU, and fines up to **€20M or 4% of global annual revenue** (whichever is higher). Data must also be kept only as long as needed, so every copy, including disaster-recovery backups in other data centers, must be deleted when the retention period ends.

**Personal data** is any information about an identified or identifiable living person, including combinations of fields that identify someone.

In [5]:
# Classification from the course exercise: is each field personal data?
fields = pd.DataFrame({
    "field": ["Genetic data", "Country", "IP address", "bob@datacamp.com", "Salary", "info@datacamp.com"],
    "personal_data": [True, True, True, True, True, False],
    "why": ["Health/biometric", "Can identify when combined", "Online identifier",
            "Named individual's email", "Linked to a person", "Generic company mailbox"],
})
fields

,field,personal_data,why
0,Genetic data,True,Health/biometric
1,Country,True,Can identify when combined
2,IP address,True,Online identifier
3,bob@datacamp.com,True,Named individual's email
4,Salary,True,Linked to a person
5,info@datacamp.com,False,Generic company mailbox


**Interpretation:** most user data a company collects is personal data, so it directly shapes how a data engineer designs storage, retention and cross-region replication.

## 5. Cloud roles

Cloud skills are in high demand, and certifications from the main providers carry weight. Existing data roles benefit from the cloud, and new roles appear (with overlap between them):

In [6]:
roles = pd.DataFrame([
    ("Data Engineer",     "Build pipelines to ingest, transform and store big data securely"),
    ("Data Analyst",      "Query cloud data with BI tools (Tableau, Power BI) and collaborate without passing files"),
    ("Data Scientist",    "Run heavy analyses faster and cheaper"),
    ("ML Scientist",      "Train and deploy (deep learning) models with large compute"),
    ("Cloud Architect",   "Translate business requirements into a scalable, cost-optimized architecture"),
    ("Cloud Engineer",    "Build, maintain and monitor cloud services; migrate from on-premise"),
    ("DevOps Engineer",   "Infrastructure as code and automation for reliability and scalability"),
    ("Security Engineer", "Define security requirements per region and protect user data"),
], columns=["role", "cloud responsibility"])
roles

,role,cloud responsibility
0,Data Engineer,"Build pipelines to ingest, transform and store..."
1,Data Analyst,"Query cloud data with BI tools (Tableau, Power..."
2,Data Scientist,Run heavy analyses faster and cheaper
3,ML Scientist,Train and deploy (deep learning) models with l...
4,Cloud Architect,Translate business requirements into a scalabl...
5,Cloud Engineer,"Build, maintain and monitor cloud services; mi..."
6,DevOps Engineer,Infrastructure as code and automation for reli...
7,Security Engineer,Define security requirements per region and pr...


## 6. The cloud infrastructure market

Choosing a provider means weighing current infrastructure costs, the cost of migrating or rebuilding apps, hiring cloud specialists, and the benefits vs. risks. A key risk is **vendor lock-in** (not being able to switch providers easily). It is reduced with multicloud/hybrid strategies, open standards and flexible contracts.

Market share at the end of 2023 (as stated in the course): **AWS 31%** (first mover, 2006), **Microsoft Azure 24%** (strong integration with Office, Windows Server and SQL Server), **Google Cloud 11%** (multicloud focus with Anthos).

In [7]:
services = pd.DataFrame({
    "AWS":          ["S3", "EC2", "RDS", "Redshift", "Kinesis", "SageMaker"],
    "Azure":        ["Blob Storage", "Virtual Machines", "SQL Database", "Data Lake Storage / Fabric",
                     "Stream Analytics", "Azure Machine Learning"],
    "Google Cloud": ["Cloud Storage", "Compute Engine", "Cloud SQL", "BigQuery", "Dataflow", "AutoML"],
}, index=["File storage", "Compute", "Relational DB", "Data warehouse / analytics",
          "Streaming", "Machine learning"])
services

,AWS,Azure,Google Cloud
File storage,S3,Blob Storage,Cloud Storage
Compute,EC2,Virtual Machines,Compute Engine
Relational DB,RDS,SQL Database,Cloud SQL
Data warehouse / analytics,Redshift,Data Lake Storage / Fabric,BigQuery
Streaming,Kinesis,Stream Analytics,Dataflow
Machine learning,SageMaker,Azure Machine Learning,AutoML


In [8]:
def provider_of(service: str) -> str:
    """Return which Big Three provider offers a given service."""
    for provider in services.columns:
        if service in services[provider].values:
            return provider
    return "Not in the table"

for s in ["BigQuery", "Kinesis", "Blob Storage", "SageMaker"]:
    print(f"{s:<13} -> {provider_of(s)}")

BigQuery      -> Google Cloud
Kinesis       -> AWS
Blob Storage  -> Azure
SageMaker     -> AWS


### Case studies (summary)

| Company | Provider | Problem | Outcome |
|---|---|---|---|
| NerdWallet | AWS SageMaker | Slow model releases | Models in days instead of months; ~75% lower training cost with on-demand instances |
| The Ottawa Hospital | Azure (Storage + Site Recovery) | Expensive on-premise disaster recovery | 700 TB backed up, compliance with health privacy law, ~50% DR savings |
| Lush | Google Cloud (Compute Engine + Cloud SQL) | E-commerce outages on Boxing Day | Zero downtime at peak, 40% lower hosting costs |

## 🎯 Key takeaways

- The cloud turns infrastructure into an on-demand, pay-per-use service; elasticity is what makes spiky workloads affordable.
- Service models define **who manages what**; deployment models define **where and with whom** the infrastructure is shared.
- Regulations like GDPR decide where data can live and how long it can be kept, including backups.
- AWS, Azure and Google Cloud offer equivalent building blocks (storage, compute, databases, warehouses, streaming, ML) under different names.

**In a Data Analyst / Data Engineer role:** this is the vocabulary behind every modern data platform: choosing S3 vs. Blob Storage for a data lake, BigQuery vs. Redshift for a warehouse, designing retention rules for personal data, and sizing pipelines that scale with demand instead of with the peak.